## Background

`output = activation(weights @ input + bias)`

In [2]:
import pandas as pd

df = pd.DataFrame({
    "运算": ["加法", "标量乘法", "矩阵乘法", "转置", "行列式", "逆", "单位矩阵"],
    "它做什么": [
        "逐元素相加",
        "缩放每个元素",
        "变换向量",
        "翻转行和列",
        "浓缩成一个数",
        "撤销一次变换",
        "什么都不做的矩阵",
    ],
    "神经网络中的用途": [
        "给输出加偏置",
        "学习率 * 梯度",
        "层的前向传播",
        "反向传播",
        "检查可逆性",
        "解线性方程组",
        "初始化、残差连接",
    ],
})
df

,运算,它做什么,神经网络中的用途
0,加法,逐元素相加,给输出加偏置
1,标量乘法,缩放每个元素,学习率 * 梯度
2,矩阵乘法,变换向量,层的前向传播
3,转置,翻转行和列,反向传播
4,行列式,浓缩成一个数,检查可逆性
5,逆,撤销一次变换,解线性方程组
6,单位矩阵,什么都不做的矩阵,初始化、残差连接


### Matrix dot multiple and matrix multiple

In [4]:
import torch

matrix_a = torch.tensor([[1, 2], [3, 4]])
matrix_b = torch.tensor([[5, 6], [7, 8]])

dot_product = matrix_a * matrix_b
matrix_product = matrix_a @ matrix_b

print(f"Dot Product: {dot_product}")
print(f"Matrix Product: {matrix_product}")


Dot Product: tensor([[ 5, 12],
        [21, 32]])
Matrix Product: tensor([[19, 22],
        [43, 50]])


## Broadcast

In [5]:
import torch

matrix_a = torch.tensor([[1, 2, 3], [4, 5, 6]])
vector_b = torch.tensor([10, 20, 30])

result = matrix_a + vector_b
print(result)

tensor([[11, 22, 33],
        [14, 25, 36]])


## Build your custom matrix/vector with core calculation

In [6]:
class Vector:
    def __init__(self, data):
        self.data = list(data)
        self.size = len(self.data)

    def __repr__(self) -> str:
        return f"Vector({self.data})"

    def __add__(self, other):
        return Vector([
            a + b for a, b in zip(self.data, other.data)
        ])

    def __sub__(self, other):
        return Vector([
            a - b for a, b in zip(self.data, other.data)
        ])

    def __mul__(self, scalar):
        return Vector([
            x * scalar for x in self.data
        ])

    def dot(self, other):
        return sum(
            a * b for a, b in zip(self.data, other.data)
        )

    def magnitude(self):
        return sum(x ** 2 for x in self.data) ** 0.5

In [7]:
from sympy.matrices import determinant


class Matrix:
    def __init__(self, data):
        self.data = [list(row) for row in data]
        self.rows = len(self.data)
        self.cols = len(self.data[0])
        self.shape = (self.rows, self.cols)

    def __repr__(self) -> str:
        rows_str = "\n ".join(str(row) for row in self.data)
        return f"Matrix({self.shape}):\n {rows_str}"

    def __add__(self, other):
        return Matrix([
            [self.data[i][j] + other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)
        ])

    def __sub__(self, other):
        return Matrix([
            [self.data[i][j] - other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)
        ]) 

    def scalar_multiply(self, scalar):
        return Matrix([
            [self.data[i][j] * scalar for j in range(self.cols)]
            for i in range(self.rows)
        ])

    def element_wise_multiply(self, other):
        return Matrix([
            [self.data[i][j] * other.data[i][j] for j in range(self.cols)]
            for i in range(self.rows)
        ])

    def matmul(self, other):
        return Matrix([
            [
                sum(self.data[i][k] * other.data[k][j] for k in range(self.cols))
                for j in range(other.cols)
            ]
            for i in range(self.rows)
        ])

    def transpose(self):
        return Matrix([
            [self.data[j][i] for j in range(self.rows)]
            for i in range(self.cols)
        ])

    def determinant(self):
        if self.shape == (1, 1):
            return self.data[0][0]
        if self.shape == (2, 2):
            return self.data[0][0] * self.data[1][1] - self.data[0][1] * self.data[1][0]
        
        det = 0
        for j in range(self.cols):
            minor = Matrix([
                [self.data[i][k] for k in range(self.cols) if k != j]
                for i in range(1, self.rows)
            ])
            det += ((-1) ** j) * self.data[0][j] * minor.determinant()
        return det

    def inverse_2x2(self):
        det = self.determinant()
        if det == 0:
            raise ValueError("Matrix is singular, no inverse exists")
        return Matrix([
            [self.data[1][1]/det, -self.data[0][1] / det],
            [-self.data[1][0]/det, self.data[0][0] / det]
        ])            

    @staticmethod
    def identity(n):
        return Matrix([
            [1 if i == j else 0 for j in range(n)]
            for i in range(n)
        ])


*Carry out some tests*

In [10]:
A = Matrix([[1, 2], [3, 4]])
B = Matrix([[5, 6], [7, 8]])

print("A + B = ", (A + B).data)
print("A - B = ", (A - B).data)
print("A * B = ", (A.element_wise_multiply(B)).data)
print("A @ B = ", (A.matmul(B)).data)
print("A.transpose() = ", (A.transpose()).data)
print("A.determinant() = ", A.determinant())
print("A.inverse_2x2() = ", A.inverse_2x2().data)

I = Matrix.identity(2)
print("A @ A^-1 =", A.matmul(A.inverse_2x2()).data)

A + B =  [[6, 8], [10, 12]]
A - B =  [[-4, -4], [-4, -4]]
A * B =  [[5, 12], [21, 32]]
A @ B =  [[19, 22], [43, 50]]
A.transpose() =  [[1, 3], [2, 4]]
A.determinant() =  -2
A.inverse_2x2() =  [[-2.0, 1.0], [1.5, -0.5]]
A @ A^-1 = [[1.0, 0.0], [0.0, 1.0]]


## Connected to Nenural Network

In [14]:
import random

inputs = Matrix([[0.5], [0.8], [0.2]])

weights = Matrix([
    [random.uniform(-1, 1) for _ in range(3)]
    for _ in range(2)
])

bias = Matrix([[0.1], [0.1]])

def relu_matrix(m):
    return Matrix([[max(0, val) for val in row] for row in m.data])

pre_activation = weights.matmul(inputs) + bias
output = relu_matrix(pre_activation)

print(f"Input shape: {inputs.shape}")
print(f"Weight shape: {weights.shape}")
print(f"Output shape: {output.shape}")
print(f"Output: {output.data}")

Input shape: (3, 1)
Weight shape: (2, 3)
Output shape: (2, 1)
Output: [[0.839530840644948], [0]]


## Replace with NumPy

In [ ]:
import numpy as np

A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print("A + B = ", A + B)
print("A - B = ", A - B)
print("A * B = ", A * B)
print("A @ B = ", A @ B)
print("A.transpose() = ", A.T)
print("A.determinant() = ", np.linalg.det(A))
print("A.inverse_2x2() = ", np.linalg.inv(A))

I = Matrix.identity(2)
print("A @ A^-1 =", A.matmul(A.inverse_2x2()).data)